# LLM-as-a-Judge | Agent-as-a-Judge (LLM-as-a-Judge)

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
candidate_model = ChatOpenAI(model="gpt-4o-mini")  # Model being evaluated
judge_model = ChatOpenAI(model="gpt-4o")  # Stronger model as judge

class JudgeState(TypedDict):
    question: str
    rubric: str
    candidate_response: NotRequired[str]
    scores: NotRequired[dict]
    overall_score: NotRequired[float]
    feedback: NotRequired[str]

In [4]:
def generate_candidate(state: JudgeState) -> dict:
    response = candidate_model.invoke(state["question"])
    return {"candidate_response": response.content}

def judge_response(state: JudgeState) -> dict:
    response = judge_model.invoke(
        f"You are an expert evaluator. Score this response on the given rubric.\n\n"
        f"Question: {state['question']}\n\n"
        f"Response to evaluate:\n{state['candidate_response']}\n\n"
        f"Rubric:\n{state['rubric']}\n\n"
        f"Score each criterion from 1-10 and provide specific feedback.\n"
        f"Return JSON:\n"
        f'{{"scores": {{"accuracy": 8, "completeness": 7, "clarity": 9}}, '
        f'"overall": 8.0, "feedback": "Detailed feedback here"}}'
    )
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
    try:
        parsed = json.loads(cleaned)
        return {
            "scores": parsed.get("scores", {}),
            "overall_score": float(parsed.get("overall", 0)),
            "feedback": str(parsed.get("feedback", "")),
        }
    except json.JSONDecodeError:
        return {"scores": {}, "overall_score": 5.0, "feedback": response.content}

In [5]:
graph = StateGraph(JudgeState)
graph.add_node("generate", generate_candidate)
graph.add_node("judge", judge_response)

graph.add_edge(START, "generate")
graph.add_edge("generate", "judge")
graph.add_edge("judge", END)

judge_pipeline = graph.compile()

In [6]:
plot_mermaid(judge_pipeline)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate(generate)
	judge(judge)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate;
	generate --> judge;
	judge --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
result = judge_pipeline.invoke({
    "question": "Explain how gradient descent works in machine learning, including variants like SGD, Adam, and learning rate scheduling.",
    "rubric": (
        "1. Accuracy: Are the technical facts correct?\n"
        "2. Completeness: Are all requested topics covered (GD, SGD, Adam, LR scheduling)?\n"
        "3. Clarity: Is the explanation clear and well-structured?\n"
        "4. Depth: Does it go beyond surface-level explanation?"
    ),
})
print(f"Overall Score: {result['overall_score']}/10")
print(f"Scores: {result['scores']}")
print(f"Feedback: {result['feedback']}")

Overall Score: 8.75/10
Scores: {'accuracy': 9, 'completeness': 9, 'clarity': 9, 'depth': 8}
Feedback: The response provides a well-structured, clear, and accurate explanation of gradient descent and its variants. The technical facts are mostly correct, though there are minor inaccuracies in describing the relationship between noisy updates and escaping local minima in SGD. The details on Adam and learning rate scheduling are comprehensive, though slight clarification on the role of each hyperparameter could further enhance depth. The content covers all requested topics, including the basic mechanics of gradient descent, stochastic gradient descent, and advanced concepts like Adam and learning rate scheduling strategies, well. Overall, the explanation is robust and informative, offering a good balance between complexity and clarity, fitting for an informed audience in machine learning. Improvements could be made with more examples or insights into why certain choices of methods or sched

In [8]:
# === Comparative Judgment (what makes Agent-as-Judge unique) ===
# Unlike Evaluator-Optimizer (#8) which iteratively improves ONE response,
# Agent-as-Judge can compare MULTIPLE candidates and pick the best — A vs B evaluation.
candidate_b = candidate_model.invoke(
    "Explain gradient descent to a 10-year-old in 3 sentences."
)

comparison = judge_model.invoke(
    f"You are judging two candidate responses to the same question.\n\n"
    f"Question: {result['question']}\n\n"
    f"Candidate A (detailed):\n{result['candidate_response']}\n\n"
    f"Candidate B (simplified):\n{candidate_b.content}\n\n"
    f"Compare both candidates on: accuracy, clarity, and completeness.\n"
    f"Declare a winner and explain why in one sentence.\n"
    f"Format: Winner: Candidate A/B — Reason: ..."
)
print(f"\n--- Comparative Judgment ---")
print(comparison.content)


--- Comparative Judgment ---
Winner: Candidate A — Reason: Candidate A provides a detailed, accurate, and complete explanation of gradient descent, including its variants and concepts like learning rate scheduling, whereas Candidate B offers a simple but less informative analogy lacking technical depth and details.


In [9]:
# Streaming

stream_invoke(
    judge_pipeline, {
        "question": "Explain how gradient descent works in machine learning, including variants like SGD, Adam, and learning rate scheduling.",
        "rubric": (
            "1. Accuracy: Are the technical facts correct?\n"
            "2. Completeness: Are all requested topics covered (GD, SGD, Adam, LR scheduling)?\n"
            "3. Clarity: Is the explanation clear and well-structured?\n"
            "4. Depth: Does it go beyond surface-level explanation?"
        ),
    }
)


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'question': 'Explain how gradient descent works in machine learning, including variants like SGD, Adam, and learning rate scheduling.',
 'rubric': '1. Accuracy: Are the technical facts correct?\n2. Completeness: Are all requested topics covered (GD, SGD, Adam, LR scheduling)?\n3. Clarity: Is the explanation clear and well-structured?\n4. Depth: Does it go beyond surface-level explanation?',
 'candidate_response': "Gradient descent is a fundamental optimization algorithm used in machine learning and deep learning to minimize a loss function by iteratively updating the parameters of a model. The basic idea is to adjust the parameters in the direction that reduces the loss, which is typically measured as a function of the parameters of the model and the training data. Here’s how gradient descent works, along with its variants and concepts like learning rate scheduling.\n\n### Basic Gradient Descent\n\n1. **Initialization**: Start with an initial set of parameters (weights) of the model, 